In [3]:
import pandas as pd
import nltk
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

# Automatically download all required NLTK tokenizer data
for resource in ['punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

# --- CONFIGURATION ---
SENTENCES_PER_PAGE = 6

In [2]:
import pandas as pd
import nltk
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
import re

# Ensure NLTK tokenizers are downloaded silently
for resource in ['punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

def generate_layout_content(text):
    """Uses LexRank to extract specific lengths of text for the Canva layout blocks."""
    if not isinstance(text, str) or len(text.strip()) < 50:
        return "Content unavailable.", "The original text was too short to summarize.", "No data."

    try:
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summarizer = LexRankSummarizer()
        
        # 1. IN BRIEF (Extract the single most important sentence)
        brief_sentences = summarizer(parser.document, 1)
        in_brief = str(brief_sentences[0]) if brief_sentences else "Summary unavailable."
        
        # 2. THE STORY (Extract the top 12 sentences for the 2-column layout)
        story_sentences = summarizer(parser.document, 4)
        the_story = " ".join([str(sentence) for sentence in story_sentences])
        
        # 3. KEY FACT (Find a high-ranking sentence containing numbers/data, excluding the In Brief)
        key_fact = ""
        for sentence in story_sentences:
            sentence_str = str(sentence)
            if re.search(r'\d+', sentence_str) and sentence_str != in_brief:
                key_fact = sentence_str
                break
                
        # Fallback if no numbers are found: just use the second most important sentence
        if not key_fact and len(story_sentences) > 1:
            key_fact = str(story_sentences[1])
            
        return in_brief, the_story, key_fact

    except Exception as e:
        print(f"  [!] Summarization Error: {e}")
        return "Error generating summary.", "Error generating story.", "Error."

def process_offline_magazine(input_filename, output_filename):
    print(f"Loading {input_filename}...")
    try:
        df = pd.read_csv(input_filename)
    except FileNotFoundError:
        print(f"Error: Could not find {input_filename}")
        return

    print(f"Processing {len(df)} articles using local NLP...\n")
    
    in_briefs = []
    the_stories = []
    key_facts = []
    
    for index, row in df.iterrows():
        print(f"[{index + 1}/{len(df)}] Extracting layout text for: {row['Title'][:40]}...")
        
        in_brief, the_story, key_fact = generate_layout_content(row.get('Original Text', ''))
        
        in_briefs.append(in_brief)
        the_stories.append(the_story)
        key_facts.append(key_fact)

    # Add the newly generated content as separate columns mapped to your PDF blocks
    df['In Brief'] = in_briefs
    df['The Story'] = the_stories
    df['Key Fact'] = key_facts
    
    # Save the polished dataset
    df.to_csv(output_filename, index=False, encoding='utf-8')
    print(f"\nSUCCESS: Local NLP complete. Saved layout-ready text to: {output_filename}")

if __name__ == "__main__":
    INPUT_FILE = "Magazine_9_Per_Category_Random.csv"
    OUTPUT_FILE = "Magazine_50_Pages_NLP_Layout.csv"
    
    process_offline_magazine(INPUT_FILE, OUTPUT_FILE)

Loading Magazine_9_Per_Category_Random.csv...
Processing 45 articles using local NLP...

[1/45] Extracting layout text for: Army, Assam Police review security frame...
[2/45] Extracting layout text for: Army Chief General Dhiraj Seth Reviews S...
[3/45] Extracting layout text for: Army Chief Reviews Peace Initiatives in ...
[4/45] Extracting layout text for: Multilateral joint military exercise in ...
[5/45] Extracting layout text for: Fake Indian Army Vehicle Busted in Manip...
[6/45] Extracting layout text for: Lieutenant Colonel Maneo Francis PF of 2...
[7/45] Extracting layout text for: Indian Army chief General Dhiraj Seth re...
[8/45] Extracting layout text for: Indian Army swiftly neutralises UXOs fol...
[9/45] Extracting layout text for: Dogs of War: K9 units of Indian Army con...
[10/45] Extracting layout text for: Army troops undertake route march at 17,...
[11/45] Extracting layout text for: After ambushes, CRPF tells forces in Man...
[12/45] Extracting layout text for: Afte

In [ ]:
import pandas as pd

def split_story_in_half(text):
    """Splits a paragraph into two equal halves without cutting words."""
    if pd.isna(text) or not isinstance(text, str):
        return "", ""
        
    words = text.split()
    midpoint = len(words) // 2
    
    part1 = " ".join(words[:midpoint])
    part2 = " ".join(words[midpoint:])
    
    return part1, part2

def format_columns_for_canva(input_filename, output_filename):
    print(f"Loading data from {input_filename}...")
    try:
        df = pd.read_csv(input_filename)
    except FileNotFoundError:
        print(f"Error: Could not find {input_filename}")
        return

    print("Splitting 'The Story' into two separate columns...")
    
    df[['Column 1', 'Column 2']] = df.apply(
        lambda row: pd.Series(split_story_in_half(row.get('The Story', ''))), 
        axis=1
    )

    df.to_csv(output_filename, index=False, encoding='utf-8')
    print(f"SUCCESS! You now have 'Column 1' and 'Column 2' ready for Canva.")
    print(f"Saved to: {output_filename}")

if __name__ == "__main__":
    INPUT_FILE = "Magazine_50_Pages_NLP_Layout.csv" 
    OUTPUT_FILE = "Magazine_45_Pages_SPLIT_COLUMNS.csv"
    
    format_columns_for_canva(INPUT_FILE, OUTPUT_FILE)

Loading data from Magazine_50_Pages_NLP_Layout.csv...
Splitting 'The Story' into two separate columns...
SUCCESS! You now have 'Column 1' and 'Column 2' ready for Canva.
Saved to: Magazine_45_Pages_SPLIT_COLUMNS.csv
